In [5]:
import requests
import pandas as pd
import pyarrow
from datetime import date
import os

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

response = requests.get(url, params={
    "format": "geojson",
    "starttime": "2025-12-01",
    "endtime": "2025-12-31",
    "minmagnitude": "4.5",
    "limit": "2000"
})

print("status:", response.status_code)



status: 200


In [6]:
data = response.json()
result = []
for feature in data['features']:
    result.append(feature['properties'])


df = pd.json_normalize(result)

# добавляем колонку бизнес даты
df['event_date'] = pd.to_datetime(df["time"], unit="ms", utc=True)
df['event_date'] = df['event_date'].dt.date

# добавляем колонку даты выгрузки
today_date = date.today()
df["update_at"] = today_date

os.makedirs("tmp", exist_ok=True)
csv_path = f"tmp/{today_date}.csv"
parquet_path = f"tmp/{today_date}.parquet"

df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False)
